In [1]:
import os

In [2]:
class TrieNode:
    def __init__(self):
        self.children = {}        # char -> TrieNode
        self.pass_count = 0       # number of words passing through this node
        self.end_count = 0        # number of words ending exactly here

In [3]:
class Trie:
    def __init__(self):
        self.root = TrieNode()
        self.total_words = 0

    def insert(self, word):
        node = self.root
        node.pass_count += 1
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
            node.pass_count += 1
        node.end_count += 1
        self.total_words += 1

    def traverse_nodes(self, word):
        """
        Traverse following the word characters and return the list of nodes
        encountered after each char.
        Example: for "cat" returns [node_after_c, node_after_a, node_after_t].
        If the path breaks (missing child), returns nodes up to break.
        """
        nodes = []
        node = self.root
        for ch in word:
            if ch in node.children:
                node = node.children[ch]
                nodes.append(node)
            else:
                break
        return nodes

    def count_prefix(self, prefix):
        """
        Return pass_count for node corresponding to prefix (i.e., number of words
        that start with prefix). If prefix not present, returns 0.
        """
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return 0
            node = node.children[ch]
        return node.pass_count

In [5]:

def valid_word(word):
    for ch in word:
      if not 'a' <= ch <= 'z':
        return False
    return True
  
def load_words(path):
    words = []
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            word = line.strip()
            if valid_word(word):
                words.append(word)
    return words

def choose_split_by_branching(trie, word):
    """
    Given trie (prefix-trie) and a word, choose a split index where maximum branching happens.
    Returns (stem, suffix, meta) where meta includes chosen index and branching/pass_count record.
    Policy:
      - Consider node positions after each char (i = 1..len-1)
      - For each node compute branching = number of children
      - Pick i maximizing (branching, pass_count_at_node) lexicographically
      - If all branching <= 1, fallback: make suffix length 1 (last char) unless word length==1
    """
    nodes = trie.traverse_nodes(word)
    L = len(nodes)
    if L == 0:
        # no traversal -> whole word unknown, treat stem empty
        return ("", word, {"index": 0, "reason": "no_trie_path"})
    # Consider split positions at node indices 0..L-2 representing split after char index i
    best_idx = None
    best_key = (-1, -1)  # (branching, pass_count)
    for i in range(L - 1):  # can't split after last char (suffix would be empty)
        node = nodes[i]
        branching = len(node.children)
        pass_count = node.pass_count
        key = (branching, pass_count)
        if key > best_key:
            best_key = key
            best_idx = i + 1  # split index: number of chars in stem
    # If no branching (all branching <=1), pick suffix length 1 as fallback
    if best_idx is None or best_key[0] <= 1:
        if len(word) == 1:
            stem, suffix = "", word
            chosen_index = 0
            reason = "fallback_short_word"
        else:
            stem, suffix = word[:-1], word[-1]
            chosen_index = len(stem)
            reason = "fallback_suffix_len_1"
    else:
        chosen_index = best_idx
        stem = word[:chosen_index]
        suffix = word[chosen_index:]
        reason = "max_branching"
    meta = {
        "index": chosen_index,
        "branching_at_split": best_key[0],
        "pass_count_at_split": best_key[1],
        "reason": reason
    }
    return stem, suffix, meta

def choose_split_by_branching_suffix_trie(suffix_trie, word):
    """
    For suffix trie, we operate on reversed word (so suffix becomes prefix).
    Returns (stem, suffix, meta) where split chosen on reversed path => suffix length = chosen_index.
    """
    rword = word[::-1]
    nodes = suffix_trie.traverse_nodes(rword)
    L = len(nodes)
    if L == 0:
        return ("", word, {"index": 0, "reason": "no_trie_path"})
    best_idx = None
    best_key = (-1, -1)
    for i in range(L - 1):
        node = nodes[i]
        branching = len(node.children)
        pass_count = node.pass_count
        key = (branching, pass_count)
        if key > best_key:
            best_key = key
            best_idx = i + 1  # number of chars in suffix
    if best_idx is None or best_key[0] <= 1:
        # fallback to suffix length 1
        if len(word) == 1:
            stem, suffix = "", word
            chosen_index = 0
            reason = "fallback_short_word"
        else:
            stem, suffix = word[:-1], word[-1]
            chosen_index = 1
            reason = "fallback_suffix_len_1"
    else:
        suffix_len = best_idx
        suffix = word[-suffix_len:]
        stem = word[:-suffix_len]
        chosen_index = suffix_len
        reason = "max_branching_reversed"
    meta = {
        "suffix_len": chosen_index,
        "branching_at_split": best_key[0],
        "pass_count_at_split": best_key[1],
        "reason": reason
    }
    return stem, suffix, meta

def suffix_frequency_in_suffix_trie(suffix_trie, suffix):
    """Return how many words (pass_count) have this suffix (use reversed traversal)."""
    if suffix == "":
        return 0
    r = suffix[::-1]
    return suffix_trie.count_prefix(r)  # reversed suffix as prefix in suffix trie

def stem_frequency_in_prefix_trie(prefix_trie, stem):
    """Return number of words that have this stem as prefix."""
    if stem == "":
        return prefix_trie.root.pass_count
    return prefix_trie.count_prefix(stem)

def analyze_words(words):
    # unique normalize
    unique_words = []
    seen = set()
    for w in words:
        if w not in seen:
            seen.add(w)
            unique_words.append(w)
    words = unique_words
    total = len(words)
    # Build tries
    prefix_trie = Trie()
    suffix_trie = Trie()
    for w in words:
        prefix_trie.insert(w)
        suffix_trie.insert(w[::-1])

    results = []
    prefix_better = 0
    suffix_better = 0
    equal_count = 0

    for w in words:
        # prefix-trie based split (stem from start)
        stem_p, suffix_p, meta_p = choose_split_by_branching(prefix_trie, w)
        # suffix-trie based split (split computed on reversed trie)
        stem_s, suffix_s, meta_s = choose_split_by_branching_suffix_trie(suffix_trie, w)

        # collect frequencies
        suffix_freq_p = suffix_frequency_in_suffix_trie(suffix_trie, suffix_p)
        suffix_prob_p = suffix_freq_p / total if total else 0.0
        stem_freq_p = stem_frequency_in_prefix_trie(prefix_trie, stem_p)
        stem_prob_p = stem_freq_p / total if total else 0.0

        suffix_freq_s = suffix_frequency_in_suffix_trie(suffix_trie, suffix_s)
        suffix_prob_s = suffix_freq_s / total if total else 0.0
        stem_freq_s = stem_frequency_in_prefix_trie(prefix_trie, stem_s)
        stem_prob_s = stem_freq_s / total if total else 0.0

        # Simple comparison metric: which method yields a more frequent suffix in dataset
        if suffix_freq_p > suffix_freq_s:
            prefix_better += 1
            better = "prefix"
        elif suffix_freq_s > suffix_freq_p:
            suffix_better += 1
            better = "suffix"
        else:
            equal_count += 1
            better = "equal"

        results.append({
            "word": w,
            "prefix_split": (stem_p, suffix_p, meta_p),
            "suffix_split": (stem_s, suffix_s, meta_s),
            "prefix_metrics": {
                "suffix_freq": suffix_freq_p,
                "suffix_prob": suffix_prob_p,
                "stem_freq": stem_freq_p,
                "stem_prob": stem_prob_p
            },
            "suffix_metrics": {
                "suffix_freq": suffix_freq_s,
                "suffix_prob": suffix_prob_s,
                "stem_freq": stem_freq_s,
                "stem_prob": stem_prob_s
            },
            "better": better
        })

    summary = {
        "total_words": total,
        "prefix_better_count": prefix_better,
        "suffix_better_count": suffix_better,
        "equal_count": equal_count,
        "prefix_trie_total_pass": prefix_trie.root.pass_count,
        "suffix_trie_total_pass": suffix_trie.root.pass_count
    }

    return results, summary

def pretty_print_and_save(results, summary, out_path="stemming_output.txt", max_print=50):
    """
    Writes a file with one result per line and prints a small sample and summary.
    Output line format:
      word = stem+suffix  [method:prefix]  suffix_freq=.. suffix_prob=.. stem_freq=.. stem_prob=..
    """
    lines = []
    for r in results:
        w = r["word"]
        sp = r["prefix_split"]
        ss = r["suffix_split"]
        pm = r["prefix_metrics"]
        sm = r["suffix_metrics"]

        line_pref = f"{w} = {sp[0]}+{sp[1]}  [method:prefix]  suf_freq={pm['suffix_freq']} suf_prob={pm['suffix_prob']:.4f} stem_freq={pm['stem_freq']} stem_prob={pm['stem_prob']:.4f}"
        line_suf = f"{w} = {ss[0]}+{ss[1]}  [method:suffix]  suf_freq={sm['suffix_freq']} suf_prob={sm['suffix_prob']:.4f} stem_freq={sm['stem_freq']} stem_prob={sm['stem_prob']:.4f}"

        # Prefer showing the method that had higher suffix frequency
        if r["better"] == "prefix":
            chosen_line = line_pref
        elif r["better"] == "suffix":
            chosen_line = line_suf
        else:
            # tie -> show both briefly
            chosen_line = line_pref + "  ||  " + line_suf

        lines.append(chosen_line)

    # write to file
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    # print a sample
    print("# Sample outputs (up to first {} words):".format(min(max_print, len(lines))))
    for ln in lines[:max_print]:
        print(ln)
    print()
    print("# Summary:")
    print(f"Total unique words processed: {summary['total_words']}")
    print(f"Prefix-method produced more common suffix for: {summary['prefix_better_count']} words")
    print(f"Suffix-method produced more common suffix for: {summary['suffix_better_count']} words")
    print(f"Ties: {summary['equal_count']} words")
    print(f"Output saved to: {out_path}")



In [7]:
input_path="/workspaces/NLP_LAB/Lab2/brown_nouns.txt" 
output_path="stemming_output.txt"
print("Loading words from:", input_path)
words = load_words(input_path)
if not words:
    print("No words found in file.")

else: 
    print(f"Loaded {len(words)} word lines (uniquified later). Building tries and analyzing...")
    results, summary = analyze_words(words)
    pretty_print_and_save(results, summary, out_path=output_path, max_print=80)

Loading words from: /workspaces/NLP_LAB/Lab2/brown_nouns.txt
Loaded 198756 word lines (uniquified later). Building tries and analyzing...
# Sample outputs (up to first 80 words):
investigation = investigati+on  [method:suffix]  suf_freq=1126 suf_prob=0.0660 stem_freq=2 stem_prob=0.0001
primary = primar+y  [method:suffix]  suf_freq=1126 suf_prob=0.0660 stem_freq=2 stem_prob=0.0001
election = electi+on  [method:suffix]  suf_freq=1126 suf_prob=0.0660 stem_freq=3 stem_prob=0.0002
evidence = evidenc+e  [method:suffix]  suf_freq=2058 suf_prob=0.1207 stem_freq=2 stem_prob=0.0001
irregularities = irregulariti+es  [method:suffix]  suf_freq=1737 suf_prob=0.1019 stem_freq=1 stem_prob=0.0001
place = plac+e  [method:suffix]  suf_freq=2058 suf_prob=0.1207 stem_freq=4 stem_prob=0.0002
jury = jur+y  [method:suffix]  suf_freq=1126 suf_prob=0.0660 stem_freq=9 stem_prob=0.0005
presentments = presentment+s  [method:suffix]  suf_freq=6324 suf_prob=0.3708 stem_freq=1 stem_prob=0.0001
charge = charg+e  [meth